# Limpeza dos dados: Futebol Feminino Internacional (1956-2026)

**Entrada:** `data/results.csv` e `data/shootouts.csv`
**Saída:** `data/results_limpo.csv` e `data/shootouts_limpo.csv`, lidos pelo notebook principal

Aqui ficam só correções de dados e verificações de integridade. Colunas derivadas (`year`, `match_id`, `total_goals`...) e decisões analíticas (como `is_competitive`, cuja lista está em `config.py`) ficam no notebook principal.

O `match_id` do notebook principal é o índice de `resultados`, então as linhas são exportadas na mesma ordem em que foram lidas.


In [1]:
import pandas as pd
from thefuzz import fuzz

### Carregamento dos Dados

Leitura de `data/results.csv` com `date` já convertida para datetime, seguida de uma inspeção estrutural inicial:

- Dimensões (partidas e colunas)
- Seleções e torneios distintos
- Linhas duplicadas
- Amostra aleatória e tipos/nulos via `info()`

In [2]:
resultados = pd.read_csv('data/results.csv', parse_dates=['date'])

print(f"Partidas contabilizadas: {resultados.shape[0]:,}")
print(f"Número de linhas: {resultados.shape[0]} | Número de colunas: {resultados.shape[1]}\n")

print(f"Seleções distintas no dataset: {pd.concat([resultados['home_team'], resultados['away_team']]).nunique()}")
print(f"Torneios distintos: {resultados['tournament'].nunique()}\n")

print(f"Total de linhas duplicadas: {resultados.duplicated().sum()}")

display(resultados.sample(5))
resultados.info()

Partidas contabilizadas: 11,650
Número de linhas: 11650 | Número de colunas: 9

Seleções distintas no dataset: 248
Torneios distintos: 121

Total de linhas duplicadas: 0


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
4268,2011-03-04,Iceland,China,2,1,Algarve Cup,Albufeira,Portugal,True
3860,2010-02-13,Japan,South Korea,2,1,EAFF Championship,Tokyo,Japan,False
765,1994-08-21,Mexico,Trinidad and Tobago,3,3,CONCACAF Championship,Montreal,Canada,True
3810,2009-11-04,Tunisia,Algeria,1,0,UNAF Tournament,Hammam Sousse,Tunisia,False
3498,2008-06-02,South Korea,Taiwan,2,0,AFC Asian Cup,Ho Chi Minh City,Vietnam,True


<class 'pandas.DataFrame'>
RangeIndex: 11650 entries, 0 to 11649
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        11650 non-null  datetime64[us]
 1   home_team   11650 non-null  str           
 2   away_team   11650 non-null  str           
 3   home_score  11650 non-null  int64         
 4   away_score  11650 non-null  int64         
 5   tournament  11650 non-null  str           
 6   city        11650 non-null  str           
 7   country     11650 non-null  str           
 8   neutral     11650 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(2), str(5)
memory usage: 739.6 KB


**Visão geral do dataset**
- Dataset com 11.650 partidas e 9 colunas, sem linhas duplicadas e sem nulos em nenhuma coluna
- 248 seleções e 121 torneios distintos
- Tipos coerentes com o conteúdo: `date` em datetime, placares em inteiro, `neutral` em booleano e demais colunas em texto

### Limpeza dos dados

In [3]:
print("--- Diagnóstico Inicial ---")

# Verificação de espaços extras nas colunas de texto
tem_espacos = (
    resultados['home_team'].str.contains(r'^\s|\s$', regex=True).any() or
    resultados['away_team'].str.contains(r'^\s|\s$', regex=True).any() or
    resultados['tournament'].str.contains(r'^\s|\s$', regex=True).any()
)
print(f"Espaços extras nas pontas dos nomes: {'Sim' if tem_espacos else 'Não'}\n")

print(f"Primeiro jogo no dataset: {resultados['date'].min()}")
print(f"Último jogo no dataset: {resultados['date'].max()}\n")

# Verificação de valores extremos nos placares
print(f"Home Score - Mínimo: {resultados['home_score'].min()} | Máximo: {resultados['home_score'].max()}")
print(f"Away Score - Mínimo: {resultados['away_score'].min()} | Máximo: {resultados['away_score'].max()}")

display(resultados[resultados['home_score'] == 24])
display(resultados[resultados['away_score'] == 24])

--- Diagnóstico Inicial ---
Espaços extras nas pontas dos nomes: Não

Primeiro jogo no dataset: 1956-09-23 00:00:00
Último jogo no dataset: 2026-06-10 00:00:00

Home Score - Mínimo: 0 | Máximo: 24
Away Score - Mínimo: 0 | Máximo: 24


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
1844,2001-12-06,North Korea,Singapore,24,0,AFC Championship,New Taipei City,Taiwan,True


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
2962,2006-11-10,Cameroon,South Africa,2,24,African Championship,Delta,Nigeria,True


**Verificação dos placares extremos**

- **Coreia do Norte 24x0 Singapura** (06/12/2001, Campeonato Asiático): confirmado na [RSSSF](https://rsssf.org/tablesa/asia-women01.html), com a mesma data do dataset.
- **Camarões 2x24 África do Sul** (10/11/2006, 3º lugar do Campeonato Africano): erro de digitação. A [RSSSF](https://rsssf.org/tablesa/afr-women06.html) registra 2-2, com a África do Sul vencendo nos pênaltis por 5-4. O placar foi corrigido para 2x2 na célula seguinte (o jogo não consta em `shootouts.csv`).
- **Polônia x Rússia** (21/09/2011, qualificatórias da Euro feminina, em Racibórz): o dataset traz 0-20, mas a [UEFA](https://www.uefa.com/womenseuro/news/newsid=1683705.html) registra 0-2 em campo (a UEFA depois deu a vitória por 3-0 à Rússia; foi mantido o placar de campo). Corrigido para 0-2 na célula seguinte.


In [4]:
# Correções verificadas em fontes externas (RSSSF e UEFA)
# 1) Camarões x África do Sul (10/11/2006, 3º lugar do Campeonato Africano): 2-2 (pên. 4-5), não 2-24
mask_placar = ((resultados['date'] == '2006-11-10') & (resultados['home_team'] == 'Cameroon')
               & (resultados['away_team'] == 'South Africa'))
resultados.loc[mask_placar, 'away_score'] = 2

# 2) Polônia x Rússia (21/09/2011, qualificatórias da Euro, Racibórz): 0-2 em campo, não 0-20
#    (a UEFA depois deu a vitória por 3-0 à Rússia; mantido o placar de campo)
mask_pol = ((resultados['date'] == '2011-09-21') & (resultados['home_team'] == 'Poland')
            & (resultados['away_team'] == 'Russia'))
resultados.loc[mask_pol, 'away_score'] = 2

# 3) OFC Championship 1998 (disputado de 9 a 17/10/1998): as partidas da 1ª e da 2ª rodada de grupos
#    estão datadas em dezembro (9 e 11/12), mas foram jogadas em 9 e 11/10
datas_ofc = {pd.Timestamp('1998-12-09'): pd.Timestamp('1998-10-09'),
             pd.Timestamp('1998-12-11'): pd.Timestamp('1998-10-11')}
mask_data = (resultados['tournament'] == 'OFC Championship') & resultados['date'].isin(datas_ofc)
resultados.loc[mask_data, 'date'] = resultados.loc[mask_data, 'date'].map(datas_ofc)

display(resultados[mask_placar | mask_pol | mask_data][['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']])


,date,home_team,away_team,home_score,away_score,tournament
1371,1998-10-09,New Zealand,Samoa,21,0,OFC Championship
1372,1998-10-09,Australia,American Samoa,21,0,OFC Championship
1377,1998-10-11,New Zealand,Fiji,14,0,OFC Championship
1378,1998-10-11,Australia,Papua New Guinea,8,0,OFC Championship
2962,2006-11-10,Cameroon,South Africa,2,2,African Championship
4488,2011-09-21,Poland,Russia,0,2,UEFA Euro qualification


In [5]:
# Maiores diferenças de placar (conferência de possíveis erros de digitação)
dif = (resultados['home_score'] - resultados['away_score']).abs()
display(resultados.assign(dif=dif).nlargest(15, 'dif')
        [['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']])

,date,home_team,away_team,home_score,away_score,tournament
1844,2001-12-06,North Korea,Singapore,24,0,AFC Championship
887,1995-09-24,China,Philippines,21,0,AFC Championship
1157,1997-12-05,Japan,Guam,21,0,AFC Championship
1307,1998-08-28,Canada,Puerto Rico,21,0,CONCACAF Championship
1371,1998-10-09,New Zealand,Samoa,21,0,OFC Championship
1372,1998-10-09,Australia,American Samoa,21,0,OFC Championship
3291,2007-10-03,Cuba,British Virgin Islands,21,0,CONCACAF Olympic Qualifying Tournament qualifi...
5045,2013-06-07,Jordan,Kuwait,21,0,AFC Asian Cup qualification
8704,2022-04-09,British Virgin Islands,Haiti,0,21,CONCACAF Championship qualification
11440,2026-04-10,Costa Rica,Cayman Islands,21,0,CONCACAF Championship qualification


**Conferência dos 15 maiores placares**

Dos 15 jogos listados acima (já com as correções), cinco foram conferidos na RSSSF e batem com o dataset: Coreia do Norte x Singapura (24-0), China x Filipinas (21-0), Japão x Guam (21-0), Nova Zelândia x Samoa (21-0) e Austrália x Samoa Americana (21-0). 

Os outros dez não foram verificados e foram mantidos, porque não há indício de erro. Nenhum é jogo de Copa do Mundo, então as análises de Copa (perguntas 3, 5 e 6) não são afetadas.

In [6]:
print("----------- Investigação de Similaridades de Seleções -----------")

def normalizar(nome):
    # Expande abreviações conhecidas antes de comparar
    nome = nome.replace('U.S.', 'United States')
    return nome.lower().strip()

todas_selecoes = pd.concat([resultados['home_team'], resultados['away_team']]).unique()
times_alfabetico = sorted(todas_selecoes)
selecoes_similares = []

# Busca por substrings entre os nomes normalizados dos times
for i, nome1 in enumerate(times_alfabetico):
    for nome2 in times_alfabetico[i+1:]:
        n1, n2 = normalizar(nome1), normalizar(nome2)
        if n1 in n2 or n2 in n1:
            selecoes_similares.append((nome1, nome2))  # guarda os nomes originais

print(f"Similaridades encontradas: {len(selecoes_similares)}")
for par in selecoes_similares:
    print(par)

----------- Investigação de Similaridades de Seleções -----------
Similaridades encontradas: 20
('American Samoa', 'Samoa')
('Congo', 'DR Congo')
('Czechoslovakia', 'Slovakia')
('Dominica', 'Dominican Republic')
('Equatorial Guinea', 'Guinea')
('FR Yugoslavia', 'Yugoslavia')
('Guinea', 'Guinea-Bissau')
('Guinea', 'Papua New Guinea')
('Hitra', 'Hitra Municipality')
('Macedonia', 'North Macedonia')
('Montenegro', 'Serbia and Montenegro')
('Netherlands', 'Netherlands Antilles')
('Niger', 'Nigeria')
('Serbia', 'Serbia and Montenegro')
('Shetland', 'Shetland Islands')
('South Sudan', 'Sudan')
('Turks and Caicos', 'Turks and Caicos Islands')
('U.S. Virgin Islands', 'United States')
('U.S. Virgin Islands', 'United States Virgin Islands')
('United States', 'United States Virgin Islands')


Dos 20 pares encontrados, a maioria são **entidades distintas e legítimas** (países vizinhos como `Niger`/`Nigeria`, territórios como `Netherlands Antilles`, sucessões políticas como `Yugoslavia`/`FR Yugoslavia`).

**Candidatos a mesma entidade com nomes diferentes (a verificar):**
- `Turks and Caicos` / `Turks and Caicos Islands`
- `Shetland` / `Shetland Islands`
- `Hitra` / `Hitra Municipality`
- `U.S. Virgin Islands` / `United States Virgin Islands`

In [7]:
print(" -------------Verificação dos candidatos a variação de nomenclatura--------------")
candidatos = [
    ('Turks and Caicos', 'Turks and Caicos Islands'),
    ('Shetland', 'Shetland Islands'),
    ('Hitra', 'Hitra Municipality'),    
    ('U.S. Virgin Islands', 'United States Virgin Islands')
]

lista_verificados = []

for nome1, nome2 in candidatos:
    datas1 = resultados[(resultados['home_team'] == nome1) | (resultados['away_team'] == nome1)]['date']
    datas2 = resultados[(resultados['home_team'] == nome2) | (resultados['away_team'] == nome2)]['date']
    
    t1_str, t2_str = "Sem dados", "Sem dados"
    m1, mx1, m2, mx2 = None, None, None, None
    sobrepoe = "N/A"
    
    if not datas1.empty:
        m1, mx1 = datas1.min(), datas1.max()
        t1_str = f"{m1.strftime('%Y-%m-%d')} a {mx1.strftime('%Y-%m-%d')}"
        
    if not datas2.empty:
        m2, mx2 = datas2.min(), datas2.max()
        t2_str = f"{m2.strftime('%Y-%m-%d')} a {mx2.strftime('%Y-%m-%d')}"
        
    if not datas1.empty and not datas2.empty:
        if max(m1, m2) <= min(mx1, mx2):
            sobrepoe = "Sim"
        else:
            sobrepoe = "Não"
            
    lista_verificados.append({
        'Nome 1': nome1,
        'Período 1': t1_str,
        'Nome 2': nome2,
        'Período 2': t2_str,
        'Sobrepõe?': sobrepoe
    })

verificados = pd.DataFrame(lista_verificados)
print(verificados.to_string(index=False))

confronto = resultados[
    ((resultados['home_team']=='U.S. Virgin Islands') & (resultados['away_team']=='United States Virgin Islands')) |
    ((resultados['home_team']=='United States Virgin Islands') & (resultados['away_team']=='U.S. Virgin Islands'))
]
print(confronto)
print(f"\nTotal de confrontos diretos: {len(confronto)}")


 -------------Verificação dos candidatos a variação de nomenclatura--------------
             Nome 1               Período 1                       Nome 2               Período 2 Sobrepõe?
   Turks and Caicos 2026-04-09 a 2026-04-16     Turks and Caicos Islands 2006-05-04 a 2023-12-04       Não
           Shetland 2005-07-10 a 2005-07-15             Shetland Islands 2025-07-13 a 2025-07-17       Não
              Hitra 2011-06-26 a 2025-07-17           Hitra Municipality 2009-06-28 a 2009-07-02       Não
U.S. Virgin Islands 2007-10-03 a 2026-04-14 United States Virgin Islands 2002-08-08 a 2025-11-27       Sim
Empty DataFrame
Columns: [date, home_team, away_team, home_score, away_score, tournament, city, country, neutral]
Index: []

Total de confrontos diretos: 0


Em todos os pares, **tudo indica a mesma entidade com grafia inconsistente**:
- **Sem sobreposição de período** (mudança de nome ao longo do tempo): `Turks and Caicos`, `Shetland`, `Hitra`
- **Com sobreposição (2002–2026)** (uso intercambiável): `U.S. Virgin Islands`
- **Sem confrontos entre os pares**

In [8]:
mapeamento_times = {
    'Shetland Islands': 'Shetland',
    'Hitra Municipality': 'Hitra',
    'Turks and Caicos Islands': 'Turks and Caicos',
    'United States Virgin Islands': 'U.S. Virgin Islands',
}

resultados['home_team'] = resultados['home_team'].replace(mapeamento_times)
resultados['away_team'] = resultados['away_team'].replace(mapeamento_times)

total_unicos_atual = pd.concat([resultados['home_team'], resultados['away_team']]).nunique()
print(f'Total de seleções únicas após a correção: {total_unicos_atual}')

Total de seleções únicas após a correção: 244


**Unificação de nomenclatura de seleções:**  Total de seleções únicas: 248 → **244**.

In [9]:
print("----------- Investigação de Similaridades de Torneios -----------")
todos_torneios = sorted(resultados['tournament'].unique())
torneios_similares = []

# Auditoria usando Fuzzy Matching (TheFuzz) para encontrar grafias parecidas
for i, nome1 in enumerate(todos_torneios):
    for nome2 in todos_torneios[i+1:]:
        score = fuzz.ratio(nome1, nome2)
        if score >= 85:
            torneios_similares.append((nome1, nome2, score))

apuramento = pd.DataFrame(torneios_similares, columns=['Torneio 1', 'Torneio 2', 'Similaridade'])
display(apuramento.sort_values('Similaridade', ascending=False).head(5))

print("----- Teste de diferenciação dos torneios 'FIFA World Cup' e 'World Cup'-----")
print(f"FIFA World Cup - Mínimo: {resultados[resultados['tournament'] == 'FIFA World Cup']['date'].dt.year.min()} | Máximo: {resultados[resultados['tournament'] == 'FIFA World Cup']['date'].dt.year.max()}")
print(f"World Cup - Mínimo: {resultados[resultados['tournament'] == 'World Cup']['date'].dt.year.min()} | Máximo: {resultados[resultados['tournament'] == 'World Cup']['date'].dt.year.max()}")

----------- Investigação de Similaridades de Torneios -----------


,Torneio 1,Torneio 2,Similaridade
27,Africa Cup of Nations qualification,African Cup of Nations qualification,99
57,Olympic qualification,Olympic qualifyication,98
23,AFF Championship,WAFF Championship,97
22,AFF Championship,SAFF Championship,97
19,AFF Championship,EAFF Championship,97


----- Teste de diferenciação dos torneios 'FIFA World Cup' e 'World Cup'-----
FIFA World Cup - Mínimo: 1991 | Máximo: 2023
World Cup - Mínimo: 1970 | Máximo: 1971


#### Verificação de similaridade nos nomes de torneios (fuzzy matching)

Comparação dos 121 torneios via `fuzz.ratio` (score ≥ 85). Resultado: 3 pares são **inconsistências genuínas**; o restante são torneios legítimos de confederações distintas, com nomenclatura parecida.

**Inconsistências a corrigir:**
- `Africa Cup of Nations qualification` / `African Cup of Nations qualification`
- `Olympic qualification` / `Olympic qualifyication`
- `Tri-Nation Cup` / `Tri-Nations Cup`

**Mantidos separados:**
- `World Cup` (1970–1971, pré-FIFA) / `FIFA World Cup` (desde 1991): torneios de natureza distinta

**Limitação:** o score bruto não distingue erro de digitação de nomenclatura legítima por confederação; validação manual segue necessária.

In [10]:
mapeamento_torneios = {
    # Correção de inconsistências de grafia / erros de digitação
    'African Cup of Nations qualification': 'Africa Cup of Nations qualification',
    'Olympic qualifyication': 'Olympic qualification',
    'Tri-Nations Cup': 'Tri-Nation Cup',
    # Renomeação para clareza (não é erro, é ambiguidade de nome)
    'World Cup': 'World Cup (pre-FIFA)',
}
resultados['tournament'] = resultados['tournament'].replace(mapeamento_torneios)
total_atual = pd.concat([resultados['tournament']]).nunique()
print(f'Total de torneios únicos após a correção: {total_atual}')

Total de torneios únicos após a correção: 118


**Unificação de nomenclatura de torneios:**  Total de torneios únicas: 121 → **118**.

**Renomeação por clareza (não é erro):** `World Cup` → `World Cup (pre-FIFA)`: jogos ocorrem todos em 1970–1971, antes da FIFA assumir a Copa do Mundo Feminina (primeiro registro de `FIFA World Cup` em 1991)

In [11]:
print("--- Validações de Integridade Estrutural ---")

print("----------Verificação de seleções jogando contra si mesmas----------")
mesmo_time = resultados[resultados['home_team'] == resultados['away_team']]
print(f'Partidas com o mesmo time: {len(mesmo_time)}')

print("----------Verificação de seleções com jogos no mesmo dia----------")
home = resultados[['date', 'home_team']].rename(columns={'home_team': 'team'})
away = resultados[['date', 'away_team']].rename(columns={'away_team': 'team'})
todas_participacoes = pd.concat([home, away])

duplicados_mesmo_dia = todas_participacoes[
    todas_participacoes.duplicated(subset=['date', 'team'], keep=False)
]
print(f'Participações duplicadas no mesmo dia: {len(duplicados_mesmo_dia)}')
display(duplicados_mesmo_dia.sort_values(by=['team', 'date']))

--- Validações de Integridade Estrutural ---
----------Verificação de seleções jogando contra si mesmas----------
Partidas com o mesmo time: 0
----------Verificação de seleções com jogos no mesmo dia----------
Participações duplicadas no mesmo dia: 10


,date,team
9631,2023-10-25,Jamaica
9636,2023-10-25,Jamaica
10322,2024-10-25,South Africa
10311,2024-10-25,South Africa
10161,2024-06-04,Székely Land
10162,2024-06-04,Székely Land
10167,2024-06-06,Sápmi
10168,2024-06-06,Sápmi
10165,2024-06-05,Tamil Eelam
10166,2024-06-05,Tamil Eelam


In [12]:
indices_para_checar = [9631, 9636, 10322, 10311]

display(resultados.loc[indices_para_checar])

resultados['fifa_affiliated'] = ~resultados['tournament'].str.contains('CONIFA', case=False)

# Conferindo
print(resultados['fifa_affiliated'].value_counts())
display(resultados[~resultados['fifa_affiliated']][['date', 'home_team', 'away_team', 'tournament']].head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
9631,2023-10-25,Panama,Jamaica,2,1,CONCACAF Gold Cup qualification,Panama City,Panama,False
9636,2023-10-25,Paraguay,Jamaica,10,0,Pan American Games,Viña del Mar,Chile,True
10322,2024-10-25,South Africa,Eswatini,3,0,COSAFA Championship,Ibhayi,South Africa,False
10311,2024-10-25,Denmark,South Africa,5,0,Friendly,Aalborg,Denmark,False


fifa_affiliated
True     11641
False        9
Name: count, dtype: int64


,date,home_team,away_team,tournament
8827,2022-07-01,Tibet,Sápmi,CONIFA World Cup
8833,2022-07-03,Sápmi,Tibet,CONIFA World Cup
10161,2024-06-04,Sápmi,Székely Land,CONIFA World Cup
10162,2024-06-04,Tamil Eelam,Székely Land,CONIFA World Cup
10165,2024-06-05,Székely Land,Tamil Eelam,CONIFA World Cup


- **Seleções com mais de uma partida no mesmo dia:** 10 participações duplicadas (5 pares)
  - `Székely Land`, `Sápmi`, `Tamil Eelam` (jun/2024): entidades CONIFA
  - `Jamaica` (2023-10-25) e `South Africa` (2024-10-25): seleções FIFA-afiliadas -- o provável é que uma das duas partidas de cada par tenha sido jogada por uma equipe diferente (base ou time misto) registrada com o nome da seleção principal.

**Nota sobre torneios CONIFA**
O dataset inclui torneios da CONIFA, confederação de seleções **não afiliadas à FIFA** (regiões, minorias, diásporas e territórios). Calendários compactos explicam partidas múltiplas em curto intervalo, sem indicar erro de dado. Métricas que assumem "seleção nacional oficial" (ex.: rankings) podem ser distorcidas se misturarem as duas categorias.

**Tratamento adotado:** registros CONIFA mantidos, com coluna auxiliar `fifa_affiliated` para filtragem conforme a análise.

### Pênaltis (`shootouts.csv`)

Mesma unificação de nomes aplicada em `resultados`, mais três validações: o vencedor é sempre uma das duas seleções, cada disputa corresponde a uma partida de `resultados` e não há disputas duplicadas.


In [13]:
penaltis = pd.read_csv('data/shootouts.csv', parse_dates=['date'])

# mesma unificação de nomes aplicada em resultados
cols_times = ['home_team', 'away_team', 'winner']
penaltis[cols_times] = penaltis[cols_times].replace(mapeamento_times)

# Validação 1: o vencedor é sempre o mandante ou o visitante
vencedor_invalido = penaltis[(penaltis['winner'] != penaltis['home_team'])
                             & (penaltis['winner'] != penaltis['away_team'])]
print(f'Disputas com vencedor fora da partida: {len(vencedor_invalido)}')

# Validação 2: cada disputa corresponde a uma partida de resultados
chave = ['date', 'home_team', 'away_team']
cruzado = penaltis.merge(resultados[chave + ['home_score', 'away_score']], on=chave, how='left', indicator=True)
print(f"Disputas sem partida correspondente em resultados: {(cruzado['_merge'] != 'both').sum()}")

# Validação 3: sem disputas duplicadas
print(f'Disputas duplicadas: {penaltis.duplicated(chave).sum()}')

assert len(vencedor_invalido) == 0 and (cruzado['_merge'] == 'both').all() and not penaltis.duplicated(chave).any()
assert len(cruzado) == len(penaltis)

# Informativo: disputas em partidas cujo placar não é empate
nao_empate = cruzado[cruzado['home_score'] != cruzado['away_score']]
print(f'\nDisputas em partida sem empate no placar: {len(nao_empate)} de {len(penaltis)}')
display(nao_empate[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'winner']])


Disputas com vencedor fora da partida: 0
Disputas sem partida correspondente em resultados: 0
Disputas duplicadas: 0

Disputas em partida sem empate no placar: 6 de 136


,date,home_team,away_team,home_score,away_score,winner
0,1984-05-27,England,Sweden,1,0,Sweden
63,2019-04-09,Congo,Gabon,0,2,Gabon
81,2021-10-26,Ethiopia,Uganda,2,0,Uganda
88,2022-02-22,Mali,Senegal,1,0,Senegal
115,2025-02-26,Zimbabwe,Angola,2,1,Angola
116,2025-02-26,Ethiopia,Uganda,2,0,Ethiopia


**Resultado:** nenhuma inconsistência. Todas as 136 disputas têm partida correspondente em `resultados`, e o vencedor é sempre uma das duas seleções.

Seis disputas ocorrem em partidas sem empate no placar (final da Euro de 1984 e qualificatórias africanas). O padrão sugere voltas de confrontos de ida e volta decididos no agregado, e elas foram mantidas.


### Exportação

Os dados limpos são salvos sem índice e sem reordenar linhas, para o `match_id` (índice de `resultados`) ser o mesmo no notebook principal. A releitura confere que datas e booleanos sobrevivem à ida e volta.


In [14]:
resultados.to_csv('data/results_limpo.csv', index=False)
penaltis.to_csv('data/shootouts_limpo.csv', index=False)

# Releitura: o arquivo salvo deve ser idêntico ao que está na memória
resultados_chk = pd.read_csv('data/results_limpo.csv', parse_dates=['date'])
penaltis_chk = pd.read_csv('data/shootouts_limpo.csv', parse_dates=['date'])
pd.testing.assert_frame_equal(resultados, resultados_chk)
pd.testing.assert_frame_equal(penaltis, penaltis_chk)

n_selecoes = pd.concat([resultados_chk['home_team'], resultados_chk['away_team']]).nunique()
n_torneios = resultados_chk['tournament'].nunique()
print(f"results_limpo.csv: {len(resultados_chk):,} partidas | {n_selecoes} seleções | {n_torneios} torneios")
print(f"colunas: {list(resultados_chk.columns)}")
print(f"shootouts_limpo.csv: {len(penaltis_chk)} disputas")
assert (len(resultados_chk), n_selecoes, n_torneios) == (11650, 244, 118)

results_limpo.csv: 11,650 partidas | 244 seleções | 118 torneios
colunas: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'fifa_affiliated']
shootouts_limpo.csv: 136 disputas
